# 06 - Predicción

Genera las predicciones finales sobre el archivo **sin etiquetar**:

1. Carga los datos sin etiquetar (`smoking_prediction_entrega.xlsx`).
2. Aplica **las mismas transformaciones** que en el entrenamiento (`utils.preparar`).
3. Genera las predicciones con el modelo y el umbral elegidos.
4. Exporta el archivo con la nueva columna **`smoking_prediction`** (valores 0 o 1) y una versión mínima con solo `id` y `smoking`.

No se reentrena nada: se usan `models/modelo_final.joblib` (XGBoost) y `models/umbral.json` tal como quedaron validados en la 05.

#### Dependencias

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import utils  # notebooks/utils.py: la misma limpieza que se aplico al entrenamiento

#### Parametros

In [2]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")

FILE_ENTREGA = RAW_DIR / "smoking_prediction_entrega.xlsx"
SALIDA_XLSX = PROCESSED_DIR / "smoking_prediction_entrega_con_prediccion.xlsx"
SALIDA_CSV = PROCESSED_DIR / "smoking_prediction_entrega_con_prediccion.csv"
SALIDA_MIN_CSV = PROCESSED_DIR / "predicciones.csv"  # version minima: id y smoking
COLUMNA_PREDICCION = "smoking_prediction"

## 1. Carga de los datos sin etiquetar

In [3]:
entrega_raw = pd.read_excel(FILE_ENTREGA)
print("Archivo sin etiquetar:", entrega_raw.shape)
print("Tiene la columna 'smoking':", "smoking" in entrega_raw.columns)
entrega_raw.head(3)

Archivo sin etiquetar: (5692, 26)
Tiene la columna 'smoking': False


,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar
0,27358,M,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,...,3.041667,0.631250,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0,Y
1,27364,M,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,...,4.208333,0.587500,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0,N
2,27368,M,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,...,2.041667,0.625694,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1,Y


## 2. Mismas transformaciones que en el entrenamiento

`utils.preparar` es la misma función que la notebook 03 aplicó a los datos de entrenamiento: corrige el error de Excel, descarta `oral`, codifica `gender` y `tartar`, recodifica `eyesight = 9.9` y corta con error si encuentra nulos o valores no vistos.

In [4]:
entrega = utils.preparar(entrega_raw)

config = json.loads((MODELS_DIR / "umbral.json").read_text())
FEATURES = config["features"]
X_entrega = entrega[FEATURES]  # mismas columnas y en el mismo orden que en el entrenamiento

print("X_entrega:", X_entrega.shape)
print("Nulos:", int(X_entrega.isna().sum().sum()))
X_entrega.head(3)

X_entrega: (5692, 24)
Nulos: 0


,gender,age,height_cm,weight_kg,waist_cm,eyesight_left,eyesight_right,hearing_left,hearing_right,systolic,...,hdl,ldl,hemoglobin,urine_protein,serum_creatinine,ast,alt,gtp,dental_caries,tartar
0,1,25,160,65,82.5,1.5,0.4,1,1,121.0,...,47.0,73.0,15.9,1,0.9,18.0,17.0,17.0,0,1
1,1,30,180,80,83.0,1.2,0.9,1,1,129.0,...,36.0,101.0,14.6,1,1.0,19.0,27.0,32.0,0,0
2,1,55,165,60,82.0,0.7,0.8,1,1,126.0,...,51.0,49.0,15.1,1,0.9,26.0,31.0,48.0,1,1


In [5]:
# Control: el resultado es identico al entrega.csv que genero la notebook 03
entrega_03 = pd.read_csv(PROCESSED_DIR / "entrega.csv")
print("Igual al procesado en la 03:", np.allclose(entrega_03[FEATURES].to_numpy(), X_entrega.to_numpy()))

Igual al procesado en la 03: True


## 3. Modelo y umbral

In [6]:
best_model = joblib.load(MODELS_DIR / "modelo_final.joblib")
UMBRAL = config["umbral"]
print("Modelo:", config["modelo"], "|", config["config"])
print("Umbral de decision:", UMBRAL)

Modelo: XGBoost | {"colsample_bytree": 0.7554, "gamma": 0.8348, "learning_rate": 0.0195, "max_depth": 4, "min_child_weight": 10, "n_estimators": 600, "reg_lambda": 0.3078, "subsample": 0.6673}
Umbral de decision: 0.49


## 4. Predicción

El modelo devuelve la probabilidad de que cada persona fume. Si es mayor o igual al umbral, se predice `1` (fuma); si no, `0`.

In [7]:
proba = best_model.predict_proba(X_entrega)[:, 1]
prediccion = (proba >= UMBRAL).astype(int)

print(pd.Series(prediccion).value_counts().rename(index={0: "no fuma (0)", 1: "fuma (1)"}).to_frame("personas"))
print(f"\n% predichos fumadores: {prediccion.mean():.1%}")

             personas
fuma (1)         3093
no fuma (0)      2599

% predichos fumadores: 54.3%


In [8]:
pd.DataFrame({"gender": entrega["gender"].map({1: "M", 0: "F"}), "prediccion": prediccion}) \
    .groupby("gender")["prediccion"].agg(["size", "mean"]).rename(columns={"size": "personas", "mean": "% predichos fumadores"}).round(3)

,personas,% predichos fumadores
gender,,
F,2075,0.000
M,3617,0.855


- Se predice **54% de fumadores**, igual que en validación (54%). Es más que el 37% real porque el modelo marca de más para no perder fumadores: eso es lo que maximiza el F1.
- Como en validación, el modelo no marca a ninguna mujer como fumadora: toda la señal viene de los hombres.

## 5. Verificaciones antes de exportar

In [9]:
salida = entrega_raw.copy()
salida[COLUMNA_PREDICCION] = prediccion

assert len(salida) == len(entrega_raw) == 5692, "Cambio la cantidad de filas"
assert salida["ID"].is_unique, "Hay IDs repetidos"
assert salida["ID"].equals(entrega_raw["ID"]), "Cambio el orden de las filas"
assert salida[COLUMNA_PREDICCION].isin([0, 1]).all(), "Hay predicciones fuera de 0/1"
assert salida[COLUMNA_PREDICCION].notna().all(), "Hay predicciones nulas"
assert list(salida.columns) == list(entrega_raw.columns) + [COLUMNA_PREDICCION], "Cambiaron las columnas"
print("Verificaciones OK:", salida.shape)

Verificaciones OK: (5692, 27)


## 6. Exportación

Se exporta el **mismo archivo recibido**, sin modificar sus columnas, con la columna `smoking_prediction` agregada al final. Se guarda en Excel (mismo formato que el original) y en CSV.

También se exporta `predicciones.csv`, con solo dos columnas: `id` y `smoking` (0 o 1).

In [10]:
salida.to_excel(SALIDA_XLSX, index=False)
salida.to_csv(SALIDA_CSV, index=False)

releido = pd.read_excel(SALIDA_XLSX)
print("Archivo exportado:", SALIDA_XLSX.name, releido.shape)
print("Predicciones identicas al releer:", releido[COLUMNA_PREDICCION].equals(salida[COLUMNA_PREDICCION]))
releido[["ID", "gender", "age", COLUMNA_PREDICCION]].head()

Archivo exportado: smoking_prediction_entrega_con_prediccion.xlsx (5692, 27)
Predicciones identicas al releer: True


,ID,gender,age,smoking_prediction
0,27358,M,25,1
1,27364,M,30,1
2,27368,M,55,1
3,27378,M,20,0
4,27381,M,25,1


In [11]:
salida_min = pd.DataFrame({"id": entrega_raw["ID"], "smoking": prediccion})

assert list(salida_min.columns) == ["id", "smoking"], "Cambiaron las columnas"
assert salida_min["id"].equals(salida["ID"]), "Cambio el orden de las filas"
assert salida_min["smoking"].isin([0, 1]).all(), "Hay predicciones fuera de 0/1"

salida_min.to_csv(SALIDA_MIN_CSV, index=False)
print("Archivo exportado:", SALIDA_MIN_CSV.name, salida_min.shape)
salida_min.head()

Archivo exportado: predicciones.csv (5692, 2)


,id,smoking
0,27358,1
1,27364,1
2,27368,1
3,27378,0
4,27381,1


## Resumen

| Paso | Resultado |
|---|---|
| Datos sin etiquetar | 5.692 filas × 26 columnas |
| Transformaciones | `utils.preparar`, la misma función del entrenamiento (idéntico a la 03) |
| Modelo | XGBoost (600 árboles de profundidad 4, `learning_rate=0,02`), umbral 0,49 |
| Predicción | 54% fumadores (`smoking_prediction = 1`) |
| Archivo | `data/processed/smoking_prediction_entrega_con_prediccion.xlsx` (y `.csv`): columnas originales + `smoking_prediction` |
| Versión mínima | `data/processed/predicciones.csv`: `id` y `smoking` |